In [ ]:
from IPython.display import display
import json
import io
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import plotly.express as px
from skimage.metrics import mean_squared_error as mse
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.util import crop
from scipy import ndimage, datasets
from scipy.ndimage import uniform_filter, gaussian_filter
from tqdm import tqdm
import sys
import yuvio
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({"axes.spines.top": False,
                     "axes.spines.right": False})

In [ ]:
def MSE(original, pred):
    mse = np.mean(np.square(original.astype(np.float64) - pred.astype(np.float64)))
    return mse

# Intro

Vamos começar olhando para os resultados dos LFs comprimidos. Nas próximas células eu quero mostrar o caminho pra chegar em cada um. Esses jsons contem diferentes métricas pra cada light field, como ssim, psnr, ...

In [ ]:
# Essa linha mostra todos os Light Fields que rodei na lfc-toolkit, com diferentes taxas de compressão,
# e passa o nome deles para txt.txt, da pra ver os 10 primeiros na próxima célula
# !ls ../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/ > txt.txt

In [ ]:
!ls ../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/ | head

In [ ]:
# essa célula abre o txt.txt, lê as linhas e passa pra uma lista os nomes
path_jpeg = '../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/'

f = open('txt.txt', 'r')
texto = f.read()
all_LFs = texto.split('\n')
all_LFs = [i for i in all_LFs if '4' in i]
all_LFs

\
Aqui eu armazeno na variável LightField_names apenas os nome dos Light Fields + suas taxas de compressão

In [ ]:
LightField_names = [lf.split('_vmaf')[0] for lf in all_LFs]
LightField_names[:10]

In [ ]:
# Nessa célula eu obtenho todos os valores do PSNR_y e SSIM
# para cada view de cada LF, que foi rodada pelo lfctoolkit.
# PS: não sei como o SSIM foi calculado
ssim_dict_of_each_LF = {}
psnr_dict_of_each_LF = {}

for each_LF, LF_name in zip(all_LFs, LightField_names):
    ssim_list_of_each_view_per_LF = []
    psnr_list_of_each_view_per_LF = []
    
    # Lembrando path_jpeg = '../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/'
    path_corrigido = os.path.join(path_jpeg + each_LF)
    
    with open(path_corrigido) as f:
        
        for linha in f:  # 1 line is exactly like '         "integer_adm_scale0": 0.888848,'
            par = linha.strip()[:-1].split(': ')  # which par is ['"integer_adm_scale0"', '0.888848']

            try:
                if par[0] == '"float_ssim"':
                    ssim_list_of_each_view_per_LF.append(float(par[1]))
                elif par[0] == '"psnr_y"':
                    psnr_list_of_each_view_per_LF.append(float(par[1]))
                    
            except:
                continue
                
    ssim_dict_of_each_LF[LF_name] = ssim_list_of_each_view_per_LF
    psnr_dict_of_each_LF[LF_name] = psnr_list_of_each_view_per_LF

mean_psnr_y_bikes015 = np.mean(psnr_dict_of_each_LF['Bikes_015'])
print('mean_psnr_y_bikes015 ', mean_psnr_y_bikes015) # verified some cells bellow

mean_ssim_bikes015 = np.mean(ssim_dict_of_each_LF['Bikes_015'])
print('mean_ssim_bikes015 ', mean_ssim_bikes015) # verified some cells below
print('\nBikes_015:', psnr_dict_of_each_LF['Bikes_015'])

\
Quero fazer um DataFrame com todos os dados, ou seja, cujas linhas sejam cada Light Field com sua respectiva taxa, e cada coluna um valor do SSIM ou do PSNR_y de cada view. Como os LFs tem quantidades de views diferentes, vou completá-los com nulos pra terem as mesmas dimensões

In [ ]:
ssim_df = pd.DataFrame.from_dict(ssim_dict_of_each_LF, orient = 'index')
ssim_df.columns = [f'view_{i}' for i in ssim_df.columns]

ssim_df['media'] = ssim_df.mean(axis = 1, skipna = True)
ssim_df['std'] = ssim_df.std(axis = 1, skipna = True)

print(ssim_df.shape)
ssim_df.index.name = 'ssim'
ssim_df[['view_0', 'view_1', 'view_2', 'view_3', 'media']].head(10)

Esse dataframe é o ssim de cada view de cada LF

\
Agora vou fazer o Data Frame pro PSNR e pro MSE, calculado diretamente do PSNR

In [ ]:
psnr_df = pd.DataFrame.from_dict(psnr_dict_of_each_LF, orient = 'index')
psnr_df.columns = [f'view_{i}' for i in psnr_df.columns]
psnr_df.index.name = 'psnr'
psnr_df['media'] = psnr_df.mean(axis = 1, skipna = True)
psnr_df[['view_0', 'view_1', 'view_2', 'view_3', 'media']].head(5)

In [ ]:
def mse_from_psnr(psnr, b):
    mse = ((2**b -1)**2)/10**(psnr/10)
    return mse

In [ ]:
mse_df = psnr_df.apply(mse_from_psnr, b = 10)
mse_df.drop(columns = ['media'])
mse_df.index.name = 'mse'
mse_df['media'] = mse_df.mean(axis = 1, skipna = True)
mse_df[['view_0', 'view_1', 'view_2', 'view_3', 'media']].head()

Pra verificar se fiz corretamente, podemos ver no json do `Bikes_015` que tanto a média do PSNR_y quanto o SSIM batem com o valor dos dfs

In [ ]:
!cd ../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/\
&& tail -n 34 Bikes_015_vmaf_11_11_625x434_yuv444p10le.json\
| head -n 5\
&& tail -n 16 Bikes_015_vmaf_11_11_625x434_yuv444p10le.json\
| head -n 5

---


## Comparando MSSIM estimado com o original
Na proxima célula eu busco por todas os jsons processados. Como não consegui passar pra um txt e fazer como fiz na primeira parte, a solução foi fazer de forma manual. Copiei e colei em uma string e depois fiz a divisão com python


In [ ]:
# !cd ../lfctest/ce/results/jplm_ssim_estimate_results/ && echo */logs/*.json

In [ ]:
_ = 'Bikes/logs/Bikes_015_11_11_625x434_PGX.json Bikes/logs/Bikes_03_11_11_625x434_PGX.json Bikes/logs/Bikes_12_11_11_625x434_PGX.json Bikes/logs/Bikes_3_11_11_625x434_PGX.json Bikes/logs/Bikes_96_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_015_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_03_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_12_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_3_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_96_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_015_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_03_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_12_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_3_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_96_11_11_625x434_PGX.json greek/logs/greek_005_9_9_512x512_PGX.json greek/logs/greek_0075_9_9_512x512_PGX.json greek/logs/greek_015_9_9_512x512_PGX.json greek/logs/greek_03_9_9_512x512_PGX.json greek/logs/greek_12_9_9_512x512_PGX.json set2/logs/set2_005_33_11_1920x1080_PGX.json set2/logs/set2_0075_33_11_1920x1080_PGX.json set2/logs/set2_015_33_11_1920x1080_PGX.json set2/logs/set2_03_33_11_1920x1080_PGX.json set2/logs/set2_12_33_11_1920x1080_PGX.json sideboard/logs/sideboard_015_9_9_512x512_PGX.json sideboard/logs/sideboard_03_9_9_512x512_PGX.json sideboard/logs/sideboard_12_9_9_512x512_PGX.json sideboard/logs/sideboard_3_9_9_512x512_PGX.json sideboard/logs/sideboard_96_9_9_512x512_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_015_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_03_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_12_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_3_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_96_11_11_625x434_PGX.json tarot/logs/tarot_005_17_17_1024x1024_PGX.json tarot/logs/tarot_0075_17_17_1024x1024_PGX.json tarot/logs/tarot_015_17_17_1024x1024_PGX.json tarot/logs/tarot_03_17_17_1024x1024_PGX.json tarot/logs/tarot_12_17_17_1024x1024_PGX.json'
LFs = _.split(' ')

path_lfc = '../lfctest/ce/results/jplm_ssim_estimate_results/'

LFs[:5]

In [ ]:
# Nessa célula eu coleto o SSIM estimado para os LFs, no geral, e não por view.
ssim_estimados_dict_por_lf = {}

for lf in LFs:

    path_corrigido = os.path.join(path_lfc + lf)     
    lf_nome = '_'.join(lf.split('/')[2].split('_')[:-4]) # Lembrando que cada lf é um elemento
    # da celula anterior. Ex: 'Bikes/logs/Bikes_03_11_11_625x434_PGX.json'
    # print(lf_nome_lista) retorna ['Bikes', '015']
    

        
    with open(path_corrigido, 'r') as f:
        
        for line in f:
            par = line.strip().split(': ')  # ['"est_ssim"', '0.9830602176303537']
            
            try:
                ssim_estimados_dict_por_lf[lf_nome] = float(par[1])
            
            except:
                continue

ssim_estimados_df = pd.DataFrame(ssim_estimados_dict_por_lf, index = ['ssim_estimados_dict_por_lf']).T
ssim_estimados_df.columns = ['ssim_estimado']
ssim_estimados_df.head()

In [ ]:
df_merged = pd.concat([ssim_estimados_df, ssim_df[['media']]], axis=1, join='outer')
df_merged.columns = ['ssim_estimado', 'ssim_original']
df_merged.head()

In [ ]:
df_merged.corr()

In [ ]:
fig = px.scatter(df_merged, x = 'ssim_estimado', y = 'ssim_original', hover_name = df_merged.index)
lo = df_merged['ssim_estimado'].min()
hi = max(df_merged['ssim_estimado'].max(), df_merged['ssim_original'].max())

fig.add_shape(type='line', x0=lo, y0=lo, x1=hi, y1=hi, line=dict(color='red', dash='dash'))
fig.show()

---

# Estimações de SSIM para Light Field

A partir daqui, vou implementar estimações de SSIM que desenvolvi para reproduzir os dados do luc e da martini no outro notebook. Algumas alterações serão feitas para acomodar o formato dos LFs.

O MSE pra cada view pode ser calculado, eu acho. Mas eu já tenho o $PSNR_y$ de cada view. Por isso, vou usar a definição de PSNR:

$$
PSNR = 10 log_{10}\frac{(2^b - 1)^2}{MSE}
$$

Logo, 

$$
MSE = \frac{(2^b-1)^2}{10^{\text{PSNR}/10}}
$$

Reescrevendo as equações anteriores em função do PSNR. A primeira equação de estimação do SSIM, deduzida pela Maria Martini é $SSIM \approx 1-\frac{MSE_l}{(2\sigma_x^2 + c_2)}$. Que fica, 

$$
SSIM \approx 1-\frac{(2^b - 1)^2}{(2\sigma_x^2 + c_2)(10^{\text{PSNR}/10})}
$$

---



onde tanto $MSE_l$ quanto $\sigma_x^2$ são locais.

A primeira equação de estimação pro $MSE_l$ com o parâmetro livre $beta$ fica:

$$MSE_l \approx MSE_G \frac{(\sigma_x^2 + \epsilon)^\beta}{\mathbb{E}[(\sigma_x^2+\epsilon)^\beta]}$$

onde $\epsilon\ =\ 1e^{-6}$ and $\beta$ ficou em 0.46 e 0.48 para cada dataset que eles usaram. Essa é outra implementação que eles fizeram. 

Como $\beta$ deu valores próximos a $0.5$, eles implementação um quarto cenário onde $\beta = 0.5$ o que resulta na variância se tornar o desvio padrão:

$$MSE_l \approx MSE_G \frac{(\sigma_x + \epsilon)}{\mathbb{E}[\sigma_x + \epsilon]}$$

Trocando na equação inicial, fica:

$$
SSIM \approx 1-\frac{MSE_G}{(2\sigma_x^2 + c_2)}\frac{(\sigma_x + \epsilon)}{\mathbb{E}[\sigma_x + \epsilon]}
$$

E o SSIM global é calculado pela média de todas as janelas do frame da imagem original.

$$
MSSIM(X, Y) = \approx \frac{1}{M}\sum_j \left ( 1-\frac{MSE_G}{(2\sigma_x^2 + c_2)}\frac{(\sigma_x + \epsilon)}{\mathbb{E}[\sigma_x + \epsilon]} \right)
$$

\
Vou começar com o Bikes. Na célula a seguir, vamos os light fields decodados. 

In [ ]:
!ls  ../lfctest/ce/results/jplm_ssim_estimate_results/Bikes/decoded/yuv/

\
A seguir está a nova função de estimar a ssim a partir da $MSE_G$ para light fields. Detalhe, global aqui quer dizer por view. Como visto antes, a MSE foi obtida do PSNR, que foi calculado usando a view inteira. Já a variância da imagem original também não foi usada. Como a martini diz no seu paper, a variância é uma das medidas que muda pouco para compressão via DCT. Nesse sentido, usei a variância local de cada view decodada.

Logo, será a implementação da seguinte função:

$$
SSIM \approx 1-\frac{MSE_G}{(2\sigma_y^2 + c_2)}
$$

In [ ]:
def calcular_ssim_por_lf(funcao_ssim_lf, LFs, filtro = 'uniform'):

    ssim_dict = {}

    for path_lf in tqdm(LFs, file=sys.stdout, colour = '#0F0'):

        elements = path_lf.split('_')
        nome = '_'.join(elements[:-4])
        n1, n2 = elements[-2].split('x')
        
        path_raw_lf = path_raw +  '_'.join(elements[:-5]) + '_' + '_'.join(elements[-4:])
        path = path_dir_yuv + path_lf
        
        LF_original = yuvio.mimread(path_raw_lf, int(n1), int(n2), 'yuv444p10le')
        LF_rec = yuvio.mimread(path, int(n1), int(n2), 'yuv444p10le')

        ssim_lista = []
    
        for i in range(len(LF_rec)):

            view_original = LF_original[i].y
            view_rec = LF_rec[i].y

            mse_g = MSE(view_original, view_rec)
            ssim_lista.append(funcao_ssim_lf(view_original, view_rec, mse_g, filtro))

        ssim_dict[nome] = ssim_lista

    df = pd.DataFrame.from_dict(ssim_dict, orient = 'index')
    
    return df

## Implementando com $MSE_G$

In [ ]:
def ssim_global_lf(view, mse_g, rec = 0, win_size = 7, b=10, k1=.01, k2=.03, filtro = 'uniform'):
    '''
    Objective: Calculate the SSIM based on the global MSE of the view and reconstructed images.
    Inputs necessary: Original image and Global MSE.
    Returns: the mean of the SSIM for the whole image.'''
    data_range = 2**b - 1
    c2 = (k2*data_range)**2

    view = view.astype(np.float64, copy=False)

    NP = win_size * win_size
    cov_norm = NP / (NP - 1)

    if filtro == 'uniform':
        ux = ndimage.uniform_filter(view, size=win_size)
        uxx = ndimage.uniform_filter(view * view, size=win_size)
        vx = np.maximum(cov_norm * (uxx - ux * ux), 0.0)

    elif filtro == 'gaussian':
        ux = ndimage.uniform_filter(view, size=win_size)
        uxx = ndimage.uniform_filter(view * view, size=win_size)
        vx = np.maximum(cov_norm * (uxx - ux * ux), 0.0)

    pad = (win_size - 1) // 2
    vx_c = crop(vx, pad)

    mssim = np.mean(1 - mse_g / (2*vx_c + c2))   # média de razões (a sua, sem bordas)

    return mssim

In [ ]:
d = {
    'Bikes': ['Bikes_015_11_11_625x434_yuv444p10le.yuv', 'Bikes_03_11_11_625x434_yuv444p10le.yuv', 'Bikes_12_11_11_625x434_yuv444p10le.yuv',
              'Bikes_3_11_11_625x434_yuv444p10le.yuv', 'Bikes_96_11_11_625x434_yuv444p10le.yuv'],
    'Danger_de_Mort': ['Danger_de_Mort_015_11_11_625x434_yuv444p10le.yuv', 'Danger_de_Mort_03_11_11_625x434_yuv444p10le.yuv', 'Danger_de_Mort_12_11_11_625x434_yuv444p10le.yuv', 
                       'Danger_de_Mort_3_11_11_625x434_yuv444p10le.yuv', 'Danger_de_Mort_96_11_11_625x434_yuv444p10le.yuv'],
    'greek': ['greek_005_9_9_512x512_yuv444p10le.yuv', 'greek_0075_9_9_512x512_yuv444p10le.yuv', 'greek_015_9_9_512x512_yuv444p10le.yuv', 
              'greek_03_9_9_512x512_yuv444p10le.yuv', 'greek_12_9_9_512x512_yuv444p10le.yuv'],
    # 'set2': ['set2_03_33_11_1920x1080_yuv444p10le.yuv', 'set2_005_33_11_1920x1080_yuv444p10le.yuv', 'set2_12_33_11_1920x1080_yuv444p10le.yuv', 
    #          'set2_015_33_11_1920x1080_yuv444p10le.yuv',  'set2_0075_33_11_1920x1080_yuv444p10le.yuv'],
    'Fountain_Vincent2': ['Fountain_Vincent2_015_11_11_625x434_yuv444p10le.yuv', 'Fountain_Vincent2_03_11_11_625x434_yuv444p10le.yuv', 
                          'Fountain_Vincent2_12_11_11_625x434_yuv444p10le.yuv', 'Fountain_Vincent2_3_11_11_625x434_yuv444p10le.yuv', 
                          'Fountain_Vincent2_96_11_11_625x434_yuv444p10le.yuv']
}

In [ ]:
dfs = []
path_raw = '/home/andrey/lfctest/ce/RAW/lightfields/YUV/serpentine/'

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_por_lf(ssim_global_lf, LFs = LFs)
    dfs.append(df)

ssim_global = pd.concat(dfs)
ssim_global['media'] = ssim_global.mean(axis = 1, skipna = True)
ssim_global['std'] = ssim_global.std(axis = 1, skipna = True)

## Implementando a estimação da Martini

In [ ]:
def ssim_martini_lf(view_original, view_rec, mse_g, filtro, win_size=7, b=10, k1=0.01, k2=0.03):
    '''
    Objective: Calculates the estimated SSIM, with the Martini estimation.
    Returns: the mean of the SSIM for the whole image. A float value'''
    data_range = 2**b - 1
    c1 = (k1 * data_range) ** 2
    c2 = (k2 * data_range) ** 2

    view_original = view_original.astype(np.float64, copy=False)
    view_rec = view_rec.astype(np.float64, copy=False)

    NP = win_size * win_size
    cov_norm = NP / (NP - 1)

    # Local means
    ux = uniform_filter(view_original, size=win_size)
    uy = uniform_filter(view_rec, size=win_size)

    # Local second moments
    uxx = uniform_filter(view_original * view_original, size=win_size)
    
    # Local variances and covariance
    vx = np.maximum(cov_norm * (uxx - ux * ux), 0.0)

    mse_l =  np.maximum(uniform_filter((view_original - view_rec) ** 2, size=win_size), 0.0)

    S = 1 - mse_l/ (2*vx + c2)

    pad = (win_size - 1) // 2

    return crop(S, pad).mean(dtype=np.float64)

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_por_lf(ssim_martini_lf, LFs = LFs)
    dfs.append(df)

ssim_martini = pd.concat(dfs)
ssim_martini['media'] = ssim_martini.mean(axis = 1, skipna = True)
ssim_martini['std'] = ssim_martini.std(axis = 1, skipna = True)

## Implementação com a $MSE_l$ estimada com $\beta \neq 0.5$

Lembrando que a equação a ser implementada é

$$MSE_l \approx MSE_G \frac{(\sigma_x^2 + \epsilon)^\beta}{\mathbb{E}[(\sigma_x^2+\epsilon)^\beta]}$$

In [ ]:
def ssim_beta_lf(original, mse_g, rec = 0, win_size=7, data_range = 255, k1=.01, k2=.03, filtro = ''):
    c2 = (k2*data_range)**2

    original = original.astype(np.float64, copy=False)

    NP = win_size * win_size
    cov_norm = NP / (NP - 1)

    ux = ndimage.uniform_filter(original, size=win_size)
    uxx = ndimage.uniform_filter(original * original, size=win_size)
    vx = np.maximum(cov_norm * (uxx - ux * ux), 0.0)

    pad = (win_size - 1) // 2
    vx_c = crop(vx, pad)

    epsilon = 1e-6
    vx_epsilon_beta = (vx_c + epsilon)**beta
    valor_esperado = np.mean(vx_epsilon_beta)
    mse_l = (mse_g/valor_esperado)*vx_epsilon_beta
    
    mssim = np.mean(1 - mse_l / (2*vx_c + c2))   # média de razões (a sua, sem bordas)

    return mssim

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_por_lf(ssim_martini_lf, LFs = LFs)
    dfs.append(df)

ssim_martini = pd.concat(dfs)
ssim_martini['media'] = ssim_martini.mean(axis = 1, skipna = True)
ssim_martini['std'] = ssim_martini.std(axis = 1, skipna = True)

## Comparando Implementações

In [ ]:
fig, ax = plt.subplots(2, 2, figsize = (10, 10))

q = [1.5, 3, 12, 30, 96]

ax[0, 0].errorbar(q, ssim_df.iloc[:5]['media'], ssim_df.iloc[:5]['std'], fmt = '-o', label = 'ssim original', color = '#000')
ax[0, 0].errorbar(q, ssim_global.iloc[:5]['media'], ssim_global.iloc[:5]['std'], fmt = '-o', label = 'ssim global Bikes', color = '#ff8000')
ax[0, 0].errorbar(q, ssim_martini.iloc[:5]['media'], ssim_martini.iloc[:5]['std'], fmt = '-o', label = 'ssim Martini Bikes', color = '#0000ff')
ax[0, 0].legend(loc = 'lower right')

ax[0, 1].errorbar(q, ssim_df.iloc[5:10]['media'], ssim_df.iloc[5:10]['std'], fmt = '-o', label = 'ssim original', color = '#000')
ax[0, 1].errorbar(q, ssim_global.iloc[5:10]['media'], ssim_global.iloc[5:10]['std'], fmt = '-o', label = 'ssim estimada Danger de Mort', color = '#ff8000')
ax[0, 1].errorbar(q, ssim_martini.iloc[5:10]['media'], ssim_martini.iloc[5:10]['std'], fmt = '-o', label = 'ssim Martini Danger de Mort', color = '#0000ff')
ax[0, 1].legend()

ax[1, 0].errorbar([0.5, 0.75, 1.5, 3, 12], ssim_df.iloc[15:20]['media'], ssim_df.iloc[15:20]['std'], fmt = '-o', label = 'ssim original', color = '#000')
ax[1, 0].errorbar([0.5, 0.75, 1.5, 3, 12], ssim_global.iloc[10:15]['media'], ssim_global.iloc[10:15]['std'], fmt = '-o', label = 'ssim estimada greek', color = '#ff8000')
ax[1, 0].errorbar([0.5, 0.75, 1.5, 3, 12], ssim_martini.iloc[10:15]['media'], ssim_martini.iloc[10:15]['std'], fmt = '-o', label = 'ssim Martini greek', color = '#0000ff')
ax[1, 0].legend()

ax[1, 1].errorbar(q, ssim_df.iloc[10:15]['media'], ssim_df.iloc[10:15]['std'], fmt = '-o', label = 'ssim original', color = '#000')
ax[1, 1].errorbar(q, ssim_global.iloc[15:20]['media'], ssim_global.iloc[15:20]['std'], fmt = '-o', label = 'ssim estimada Fountain Vincent2', color = '#ff8000')
ax[1, 1].errorbar(q, ssim_martini.iloc[15:20]['media'], ssim_martini.iloc[15:20]['std'], fmt = '-o', label = 'ssim Martini Fountain Vincent2', color = '#0000ff')
ax[1, 1].legend();